# 🏇 경마픽 운영 런북 (실행 가능한 가이드)

**커널: Deno** — 각 상황은 `왜 그런지` 설명 + `상태 확인 셀`(안전) + `실행 셀`(⚠️) 로 구성됩니다.

## 규약
- ✅ **상태 확인 셀**은 언제든 실행해도 안전합니다 (읽기 전용)
- ⚠️ **실행 셀**은 실제로 데이터를 만들고 **커밋·push(=배포)까지** 합니다 — 주석(`//`)을 풀어야 동작
- 원칙: 결과 확정 경주의 예측은 불변(`--force` 금지) · `validate` 실패 시 커밋 금지

In [1]:
// 셸 헬퍼 — 리포 루트를 찾아 명령을 실행하고 출력을 표시한다
async function findRepo(start = Deno.cwd()): Promise<string> {
  let dir = start;
  while (true) {
    try { await Deno.stat(`${dir}/Makefile`); return dir; } catch { /* 계속 */ }
    const parent = dir.replace(/\/[^/]+$/, "");
    if (parent === dir || parent === "") throw new Error("리포 루트를 못 찾음");
    dir = parent;
  }
}
const REPO = await findRepo();

async function sh(cmd: string): Promise<number> {
  const proc = new Deno.Command("bash", {
    args: ["-lc", cmd], cwd: REPO, stdout: "piped", stderr: "piped",
  });
  const out = await proc.output();
  const text = new TextDecoder().decode(out.stdout) + new TextDecoder().decode(out.stderr);
  console.log(text.trim() || "(출력 없음)");
  return out.code;
}
console.log("리포:", REPO);

리포: /home/rhgw/code/r/r-8282


---
## 상황 ① 예측을 수동으로 (다시) 돌리고 싶다

**언제**: 모델을 새로 머지했을 때(예: v2), 기수변경이 많을 때, 아침 타이머가 노트북 꺼짐으로 못 돌았을 때.

**원리**: 타이머가 실행하는 것은 `scripts/ops.sh predict` — main 동기화 → `make predict`(활성 모델로 스코어링+AI 보정) → `validate` 게이트 → **변경이 있을 때만** 커밋·push(Vercel 자동 배포).
결과가 이미 확정된 경주는 자동으로 건너뛰므로(예측 불변 원칙) 몇 번을 돌려도 안전합니다.
쓰기는 멱등이라(타임스탬프만 달라지면 무변경) 내용이 같으면 커밋도 생기지 않습니다.

In [2]:
// ✅ 상태 확인: 타이머·최근 실행·데이터 변경 여부
await sh("systemctl --user list-timers 'kyongma-*' --no-pager | head -5");
await sh("journalctl --user -u kyongma-predict -n 8 --no-pager 2>/dev/null || echo '(예측 실행 기록 없음)'");
await sh("git status --short data/ | head -5; true");

NEXT                          LEFT LAST                              PASSED UNIT                  ACTIVATES
Sun 2026-08-16 19:34:38 KST     8h -                                      - kyongma-results.timer kyongma-results.service
Fri 2026-08-21 07:34:45 KST 4 days Sun 2026-08-16 07:33:36 KST 2h 36min ago kyongma-predict.timer kyongma-predict.service

2 timers listed.


8월 16 07:38:58 rhgw ops.sh[1174406]: WARNING kra_predict.ai: seoul 7경주 AI 분석 실패(1/2): Expecting value: line 17 column 28 (char 916)
 8월 16 07:46:57 rhgw ops.sh[1174406]: AI 분석: 성공 16경주 · 통계 단독 폴백 1경주
 8월 16 07:46:57 rhgw ops.sh[1174406]: 예측 생성 17/17경주 · 기록 17건 (변경 17건)
 8월 16 07:46:57 rhgw ops.sh[1174406]: git diff로 data/ 변경을 검토한 뒤 커밋·push 하세요.
 8월 16 07:47:00 rhgw ops.sh[1192373]: ✓ data/ 검증 통과
 8월 16 07:47:04 rhgw ops.sh[1174271]: push 완료 — Vercel 자동 배포 트리거됨
 8월 16 07:47:04 rhgw systemd[4097]: Finished kyongma-predict.service - 경마픽 예측 자동 생성 (당일 개최 경주).
 8월 16 07:47:04 rhgw systemd[4097]: kyongma-predict.service: Consumed 1min 42.850s CPU time.


(출력 없음)


0

In [3]:
// ⚠️ 실행 A(권장): 타이머와 완전히 같은 경로 — pull→predict→validate→커밋·push까지 한 번에
// await sh("systemctl --user start kyongma-predict.service && sleep 3 && journalctl --user -u kyongma-predict -n 15 --no-pager");

// ⚠️ 실행 B(세밀 제어): 단계별 수동 — diff를 눈으로 검토하고 싶을 때
// await sh("make predict DATE=$(date +%F) FLAGS='--refresh --ai-model haiku'");
// await sh("git diff --stat data/");
// await sh("git add data/ && git commit -m 'data: 수동 예측 재생성' && git push");

> `--refresh`를 붙이는 이유: 아침에 이미 한 번 돌았다면 raw 캐시가 그 시점 응답(기수변경 전, 마체중 미발표 등)을 물고 있습니다. 캐시를 무시하고 새로 받아야 최신 출전 정보가 반영됩니다.

---
## 상황 ② 경주 결과를 반영하고 싶다 / 어제 미확정 경주가 남았다

**원리**: `results`는 결과종합+확정배당을 받아 경주 파일에 결과만 붙이고(예측 불변) 적중률을 재계산합니다.
당일 저녁엔 마지막 1~2경주가 '순위 미확정'일 수 있어, 타이머의 results는 **오늘+어제를 함께 스윕**합니다.
수동 실행 시엔 `--refresh` 필수 — 캐시가 미확정 상태를 기억하고 있기 때문입니다.

In [4]:
// ✅ 상태 확인: 미확정 경주(예측만 있고 결과 없는 경주)가 있는지
await sh(`python3 - <<'EOF'
import json, pathlib
for p in sorted(pathlib.Path('data/meets').glob('*/*/r*.json')):
    r = json.loads(p.read_text())
    if r['prediction'] and not r['result'] and not r['canceled']:
        print('미확정:', r['date'], r['track'], f"{r['raceNo']}경주")
EOF`);

미확정: 2026-08-16 busan 1경주
미확정: 2026-08-16 busan 2경주
미확정: 2026-08-16 busan 3경주
미확정: 2026-08-16 busan 4경주
미확정: 2026-08-16 busan 5경주
미확정: 2026-08-16 busan 6경주
미확정: 2026-08-16 busan 7경주
미확정: 2026-08-16 seoul 1경주
미확정: 2026-08-16 seoul 2경주
미확정: 2026-08-16 seoul 3경주
미확정: 2026-08-16 seoul 4경주
미확정: 2026-08-16 seoul 5경주
미확정: 2026-08-16 seoul 6경주
미확정: 2026-08-16 seoul 7경주
미확정: 2026-08-16 seoul 8경주
미확정: 2026-08-16 seoul 9경주
미확정: 2026-08-16 seoul 10경주


0

In [5]:
// ⚠️ 실행: 타이머 경로 그대로 (오늘+어제 스윕)
// await sh("systemctl --user start kyongma-results.service && sleep 3 && journalctl --user -u kyongma-results -n 15 --no-pager");

// ⚠️ 특정 날짜만:
// await sh("make results DATE=2026-08-16 FLAGS='--refresh' && git add data/ && git commit -m 'data: 결과 반영' && git push");

---
## 상황 ③ 자동화가 살아있는지 확인하고 싶다

- 타이머 일정: 예측 금·토·일 07:30 / 결과 금·토·일 19:30 + 월 10:00 (KST)
- 노트북이 꺼져 있었다면 `Persistent=true` 덕분에 **부팅 후 자동 보충 실행**됩니다
- 웹에서도 확인 가능: `/admin` (실행 이력·타이머 생존 배지 — Supabase 텔레메트리)

In [6]:
// ✅ 상태 확인
await sh("systemctl --user list-timers 'kyongma-*' --no-pager");
await sh("journalctl --user -u kyongma-predict -u kyongma-results --since '-3 days' --no-pager | tail -12");

NEXT                          LEFT LAST                              PASSED UNIT                  ACTIVATES
Sun 2026-08-16 19:34:38 KST     8h -                                      - kyongma-results.timer kyongma-results.service
Fri 2026-08-21 07:34:45 KST 4 days Sun 2026-08-16 07:33:36 KST 2h 36min ago kyongma-predict.timer kyongma-predict.service

2 timers listed.
Pass --all to see loaded but inactive timers, too.


8월 16 07:33:40 rhgw ops.sh[1174372]: Uninstalled 1 package in 1ms
 8월 16 07:33:40 rhgw ops.sh[1174372]: Installed 1 package in 3ms
 8월 16 07:35:34 rhgw ops.sh[1174406]: WARNING kra_predict.ai: seoul 3경주 AI 분석 실패(1/2): Expecting value: line 19 column 28 (char 1080)
 8월 16 07:36:33 rhgw ops.sh[1174406]: WARNING kra_predict.ai: seoul 3경주 AI 분석 실패(2/2): Expecting value: line 19 column 28 (char 1044)
 8월 16 07:38:58 rhgw ops.sh[1174406]: WARNING kra_predict.ai: seoul 7경주 AI 분석 실패(1/2): Expecting value: line 17 column 28 (char 916)
 8월 16 07:46:57 rhgw ops.sh[1174406]: AI 분석: 성공 16경주 · 통계 단독 폴백 1경주
 8월 16 07:46:57 rhgw ops.sh[1174406]: 예측 생성 17/17경주 · 기록 17건 (변경 17건)
 8월 16 07:46:57 rhgw ops.sh[1174406]: git diff로 data/ 변경을 검토한 뒤 커밋·push 하세요.
 8월 16 07:47:00 rhgw ops.sh[1192373]: ✓ data/ 검증 통과
 8월 16 07:47:04 rhgw ops.sh[1174271]: push 완료 — Vercel 자동 배포 트리거됨
 8월 16 07:47:04 rhgw systemd[4097]: Finished kyongma-predict.service - 경마픽 예측 자동 생성 (당일 개최 경주).
 8월 16 07:47:04 rhgw systemd[4097]: kyo

0

---
## 상황 ④ 모델을 재학습하거나 백테스트하고 싶다

**철칙: 학습 기간과 평가(백테스트/공개) 기간은 절대 겹치면 안 됩니다.** 현재 v2는 2025-07~2026-06 학습, 2026-07~08 평가.

- 재학습: `kra-predict train --from YYYY-MM --to YYYY-MM` → `weights_v2.json` 갱신 (이 파일이 있으면 predict가 자동으로 v2, 지우면 v1 폴백)
- 백테스트: `kra-predict backtest --months YYYY-MM,YYYY-MM` → `data/stats/backtest.json` → `/model` 페이지에 버전별 공개

In [7]:
// ✅ 현재 활성 모델과 학습 정보 확인
await sh("python3 -c \"import json; d=json.load(open('pipeline/kra_predict/weights_v2.json')); print('버전:', d['version'], '| 학습:', d['trainFrom'], '~', d['trainTo'], f\\\"({d['trainRaces']}경주)\\\"); print('β:', d['beta'])\"");

버전: v2 | 학습: 2025-07 ~ 2026-06 (2129경주)
β: {'winRate1y': 0.580267, 'placeRate1y': 1.20225, 'rating': 0.253317, 'jockeyWinRate': 0.79476, 'trainerWinRate': 0.407128, 'bodyWeightStability': 0.03578, 'rest': 0.103931}


0

In [8]:
// ⚠️ 재학습·백테스트 (수 분 소요, API 쿼터 사용 — 캐시가 있으면 무비용)
// await sh("cd pipeline && uv run kra-predict train --from 2025-07 --to 2026-06");
// await sh("cd pipeline && uv run kra-predict backtest --months 2026-07,2026-08");
// await sh("git add data/ pipeline/kra_predict/weights_v2.json && git commit -m 'model: 재학습' && git push");

---
## 상황 ⑤ 뭔가 이상하다 (트러블슈팅)

| 증상 | 원인 | 처치 |
|---|---|---|
| `HTTP 403 (…)` 로그 | 해당 API 활용신청 미승인 또는 키 문제 | data.go.kr 마이페이지에서 승인 확인. 미승인 API는 자동으로 빈 목록 처리되므로 파이프라인은 계속 동작 |
| `가용한 세션이 존재하지 않습니다` | KRA 구세대 API 서버 혼잡(일시) | 자동으로 보강 생략됨 — 다음 실행에서 회복 |
| `AI 분석 실패 → 폴백` | claude 응답 파싱 실패/타임아웃 | 정상 설계 — 통계 단독으로 완주. 반복되면 `claude` 로그인 상태 확인 |
| `validate 실패` | 스키마 위반 데이터 | **커밋 금지.** zod/schema 가 원천 — 코드 수정 후 재실행 |
| push 실패 | 네트워크/인증 | 커밋은 로컬에 남음 — 다음 자동 실행이 재시도 |

In [9]:
// ✅ 종합 헬스체크
await sh("cd pipeline && uv run kra-predict validate");
await sh("cd pipeline && uv run pytest -q 2>&1 | tail -2");

✓ data/ 검증 통과
